# 01 — Audit data temporal fusion (CPU)
**Fase P0, belum training fusion.** Tidak perlu GPU; jalankan berurutan.
Membaca folder sumber Drive A berdasarkan ID, bukan isi akun login secara umum.
Login harus mempunyai izin ke sumber itu. Tidak mengubah dataset/checkpoint lama.
Unduhan maksimum: NPZ 400 MiB + enam PCAP masing-masing 8 MiB.
Pemeriksaan PCAP bersifat sampel; kecocokan hash bukan bukti pairing lengkap.
Upload laporan hanya di cell terakhir. `training_ready=false` adalah gerbang ilmiah,
bukan tanda hasil replikasi gagal. Rujukan dan tahap lanjut ada di README/PLAN folder ini.


In [ ]:
%pip -q install scapy==2.5.0
import sys, json, hashlib, io, uuid
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
auth.authenticate_user()
credentials, _ = google.auth.default()
drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
WORK = Path('/content/temporal_pilot_p0') / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
print('CPU audit run:', RUN_ID)


In [ ]:
# Snapshot source; no moving GitHub branch dependency.
SOURCES = {'data/Preprocessing/utils.py': 'import numpy as np\nimport binascii\nimport scapy.all as scapy\n\n\nPACKETS_PER_FLOW = 8\nHEADER_BYTES_PER_PACKET = 80\nPAYLOAD_BYTES_PER_PACKET = 48\nBYTES_PER_PACKET = HEADER_BYTES_PER_PACKET + PAYLOAD_BYTES_PER_PACKET\nIMAGE_SIDE = 32\nIMAGE_BYTES = IMAGE_SIDE * IMAGE_SIDE\n\n\n# hex to 0-255\ndef string_to_hex_array(flow_string):\n    return np.array([int(flow_string[i:i + 2], 16) for i in range(0, len(flow_string), 2)])\n\n\ndef read_pcap_list(pcap_filename, if_augment=False, remove_ip=True, keep_payload=True):\n    """Convert eight packets into the paper\'s 32x32 grayscale input."""\n    header_hex_length = HEADER_BYTES_PER_PACKET * 2\n    payload_hex_length = PAYLOAD_BYTES_PER_PACKET * 2\n    packets = scapy.rdpcap(pcap_filename)\n    data = []\n    flow_hex_length = IMAGE_BYTES * 2\n    for packet in packets:\n        try:\n            header, payload = raw_packet_to_string(packet, remove_ip=remove_ip, keep_payload=keep_payload)\n        except ValueError:\n            # Excluded packets do not consume one of the first eight IP slots.\n            continue\n        data.append(header + payload)\n        if not if_augment and len(data) == PACKETS_PER_FLOW:\n            break\n\n    if not data:\n        return []\n\n    if not if_augment or len(data) <= PACKETS_PER_FLOW:\n        flow_string = \'\'.join(data)\n        flow_string += \'0\' * (flow_hex_length - len(flow_string))\n        flow_array = string_to_hex_array(flow_string)\n        return [{\n            "data": flow_array,\n        }]\n    else:\n        assert len(data) > PACKETS_PER_FLOW\n        flow_array_list = []\n        for i in range(len(data) - PACKETS_PER_FLOW + 1):\n            flow_string = \'\'.join(data[i:i + PACKETS_PER_FLOW])\n            flow_array_list.append(string_to_hex_array(flow_string))\n        return [{\n            "data": flow_array,\n        } for flow_array in flow_array_list]\n\n\ndef raw_packet_to_string(packet, remove_ip=True, keep_payload=True):\n    """Keep network/transport headers and application bytes as separate regions.\n\n    Read the transport payload structurally, including decoded application layers;\n    a Scapy Raw layer is not required. Never modify the caller\'s captured packet.\n    """\n    header_hex_length = HEADER_BYTES_PER_PACKET * 2\n    payload_hex_length = PAYLOAD_BYTES_PER_PACKET * 2\n    if scapy.IP in packet:\n        ip = packet[scapy.IP].copy()\n        if ip.frag or ip.flags.MF:\n            raise ValueError(\'Fragmented packets require reassembly before extraction\')\n        pad_address = \'0.0.0.0\'\n    elif scapy.IPv6 in packet:\n        ip = packet[scapy.IPv6].copy()\n        if scapy.IPv6ExtHdrFragment in ip:\n            raise ValueError(\'Fragmented packets require reassembly before extraction\')\n        pad_address = \'::\'\n    else:\n        raise ValueError(\'Non-IP packet\')\n    if scapy.TCP in ip:\n        transport = ip[scapy.TCP]\n    elif scapy.UDP in ip:\n        transport = ip[scapy.UDP]\n        if transport.sport in (67, 68, 546, 547) or transport.dport in (67, 68, 546, 547):\n            raise ValueError(\'DHCP excluded\')\n    else:\n        raise ValueError(\'Expected a TCP or UDP flow\')\n    # Remove only capture padding, not application data or transport options.\n    if scapy.Padding in ip:\n        ip[scapy.Padding].underlayer.remove_payload()\n    application_bytes = bytes(transport.payload)\n    if remove_ip:\n        ip.src, ip.dst = pad_address, pad_address\n    network_bytes = bytes(ip)\n    header_length = len(network_bytes) - len(application_bytes)\n    header = network_bytes[:header_length].hex()\n    payload = application_bytes.hex() if keep_payload else \'\'\n    header = (\n        header[:header_hex_length]\n        if len(header) > header_hex_length\n        else header + \'0\' * (header_hex_length - len(header))\n    )\n    payload = (\n        payload[:payload_hex_length]\n        if len(payload) > payload_hex_length\n        else payload + \'0\' * (payload_hex_length - len(payload))\n    )\n    return header, payload\n\n\n\nif __name__ == "__main__":\n    pass\n\n', 'pilot/audit.py': '"""CPU-only evidence audit. Hash matches are candidates, never pairing proof."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef image_hash(image):\n    image = np.asarray(image)\n    if image.size != 1024 or image.dtype != np.uint8:\n        raise ValueError(\'Expected exactly 1024 uint8 image bytes; no silent conversion\')\n    return hashlib.sha256(image.tobytes(order=\'C\')).hexdigest()\n\n\ndef audit_npz(path, index):\n    path = Path(path)\n    with np.load(path, allow_pickle=False) as data:\n        keys = list(data.files)\n        x, y = data[\'data\'], data[\'target\']\n        if x.ndim < 2 or np.prod(x.shape[1:]) != 1024 or x.dtype != np.uint8:\n            raise ValueError(f\'{path.name}: unsupported image schema {x.shape}/{x.dtype}\')\n        if y.ndim != 1 or len(x) != len(y) or not np.issubdtype(y.dtype, np.integer):\n            raise ValueError(f\'{path.name}: invalid integer labels\')\n        for row, (image, label) in enumerate(zip(x, y)):\n            key = image_hash(image)\n            entry = index.setdefault(key, {\'count\': 0, \'examples\': [], \'labels\': set()})\n            entry[\'count\'] += 1\n            entry[\'labels\'].add(int(label))\n            if len(entry[\'examples\']) < 4:\n                entry[\'examples\'].append({\'file\': path.name, \'row\': row, \'label\': int(label)})\n        labels, counts = np.unique(y, return_counts=True)\n        return {\'file\': path.name, \'keys\': keys, \'shape\': list(x.shape),\n                \'dtype\': str(x.dtype), \'samples\': len(x),\n                \'class_counts\': {str(k): int(v) for k, v in zip(labels, counts)},\n                \'file_sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                \'explicit_flow_id_present\': any(k in keys for k in (\'flow_id\', \'pcap_id\', \'sample_id\'))}\n\n\ndef audit_pcap(path, index, packet_limit=10000):\n    from scapy.all import IP, IPv6, TCP, UDP, Padding, PcapReader\n    from data.Preprocessing.utils import raw_packet_to_string\n\n    path = Path(path)\n    flows, blocks, seq = set(), [], []\n    first_endpoint = previous_time = None\n    valid = excluded = scanned = 0\n    complete = True\n    # Streaming: never rdpcap() a multi-gigabyte capture.\n    with PcapReader(str(path)) as packets:\n        for packet in packets:\n            if scanned >= packet_limit:\n                complete = False\n                break\n            scanned += 1\n            try:\n                header, payload = raw_packet_to_string(packet)\n            except ValueError:\n                excluded += 1\n                continue\n            ip = packet[IP] if IP in packet else packet[IPv6]\n            transport = ip[TCP] if TCP in ip else ip[UDP]\n            src, dst = (ip.src, int(transport.sport)), (ip.dst, int(transport.dport))\n            flow = (ip.version, \'TCP\' if TCP in ip else \'UDP\', tuple(sorted((src, dst))))\n            flows.add(flow)\n            valid += 1\n            if len(flows) > 1:\n                # A capture is not a flow; refuse to fuse unrelated conversations.\n                return {\'file\': path.name, \'status\': \'MULTIFLOW_NEEDS_SESSIONIZATION\',\n                        \'scanned_packets\': scanned, \'observed_biflows_at_least\': len(flows),\n                        \'pairing_verified\': False}\n            if len(blocks) < 8:\n                if first_endpoint is None:\n                    first_endpoint = src\n                timestamp = float(packet.time)\n                iat = 0.0 if previous_time is None else timestamp - previous_time\n                if not np.isfinite(timestamp) or not np.isfinite(iat) or iat < 0:\n                    return {\'file\': path.name, \'status\': \'INVALID_TIMESTAMPS\', \'pairing_verified\': False}\n                transport_copy = transport.copy()\n                if Padding in transport_copy:\n                    transport_copy[Padding].underlayer.remove_payload()\n                seq.append([len(bytes(transport_copy.payload)), int(src != first_endpoint), iat])\n                blocks.append(header + payload)\n                previous_time = timestamp\n    result = {\'file\': path.name, \'scanned_packets\': scanned, \'valid_packets\': valid,\n              \'excluded_packets\': excluded, \'capture_fully_scanned\': complete,\n              \'pairing_verified\': False}\n    if not blocks:\n        return dict(result, status=\'NO_VALID_PACKETS\')\n    if not complete:\n        return dict(result, status=\'SCAN_LIMIT_NO_PAIRING_CLAIM\')\n    image = np.frombuffer(bytes.fromhex(\'\'.join(blocks)).ljust(1024, b\'\\x00\'), dtype=np.uint8)\n    key = image_hash(image)\n    match = index.get(key, {\'count\': 0, \'examples\': [], \'labels\': set()})\n    return dict(result, status=\'HASH_CANDIDATE_ONLY\' if match[\'count\'] else \'NO_HASH_MATCH\',\n                image_sha256=key, matching_rows=match[\'count\'], examples=match[\'examples\'],\n                sequence_length=len(seq), sequence_features=[\'transport_payload_bytes\', \'direction\', \'iat_seconds\'],\n                first_iat_seconds=seq[0][2],\n                note=\'A single biflow tuple does not prove a unique session. Check provenance, labels, windows and splits.\')\n\n\ndef audit_paths(npz_paths, pcap_paths=(), packet_limit=10000):\n    index, report = {}, {\'schema\': \'temporal-pilot-audit-v1\', \'datasets\': [], \'pcaps\': [], \'errors\': []}\n    for path in npz_paths:\n        try:\n            report[\'datasets\'].append(audit_npz(path, index))\n        except Exception as exc:\n            report[\'errors\'].append({\'file\': Path(path).name, \'error\': str(exc)})\n    for path in pcap_paths:\n        try:\n            report[\'pcaps\'].append(audit_pcap(path, index, packet_limit))\n        except Exception as exc:\n            report[\'errors\'].append({\'file\': Path(path).name, \'error\': str(exc)})\n    report[\'unique_image_hashes\'] = len(index)\n    # Dataset label namespaces differ; do not compare labels across NPZ datasets.\n    report[\'training_ready\'] = False\n    report[\'next_step\'] = (\'Build and verify a full paired manifest: source capture/session/window, image hash, \'\n                           \'packet indices/times, label mapping and leakage-safe split. \'\n                           \'A sampled hash audit alone never authorizes training.\')\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--dataset-dir\', type=Path, required=True)\n    parser.add_argument(\'--pcap\', type=Path, action=\'append\', default=[])\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    report = audit_paths(sorted(args.dataset_dir.glob(\'*.npz\')), args.pcap)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(report, indent=2))\n', 'pilot/__init__.py': '"""Isolated temporal-fusion pilot; does not modify the replication pipeline."""\n'}
SOURCE_SHA256 = {'data/Preprocessing/utils.py': '4bb13d662c25a2f4086a475791640b580e090cc1253ae7ce1f08c200927752d4', 'pilot/audit.py': '90468d47ad4b1ec858ba1b581db449df46f34f886e64615fa71654c865c3900e', 'pilot/__init__.py': 'fa09be3e956caf07db20228406db1ba733bf62f6a7dd665e1c66fb6fbcd95ef4'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_SHA256[name]
    path = WORK / 'src' / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(WORK / 'src'))
from pilot.audit import audit_paths
PILOT_FOLDER = '1eRt_MeJkoVvFCMTfuHssELQ2RgqQK0Ii'
DATA_FOLDER = '1vCeFiN3ng82p9_p0tfnvwzjCe_pdeMie'
RAW_FOLDER = '1-fYdLsJfBVV5rTx922y25aRQA0k_fSB0'
for folder in (PILOT_FOLDER, DATA_FOLDER, RAW_FOLDER):
    meta = drive.files().get(fileId=folder, fields='id,name,mimeType', supportsAllDrives=True).execute()
    assert meta['mimeType'] == 'application/vnd.google-apps.folder'
    print('Akses OK:', meta['name'], meta['id'])


In [ ]:
# Limited Drive enumeration; skipped/failed entries remain visible in report.
FOLDER_MIME = 'application/vnd.google-apps.folder'
def children(folder, page_token=None):
    return drive.files().list(q=f"'{folder}' in parents and trashed=false",
        fields='nextPageToken,files(id,name,mimeType,size,md5Checksum)',
        pageSize=100, pageToken=page_token, orderBy='name',
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()

def inventory(root, max_entries=3000, max_folders=100):
    queue, rows, errors = deque([(root, '')]), [], []
    visited = set()
    partial = False
    while queue:
        if len(visited) >= max_folders or len(rows) >= max_entries:
            partial = True
            break
        folder, prefix = queue.popleft()
        if folder in visited:
            continue
        visited.add(folder)
        token = None
        try:
            while True:
                response = children(folder, token)
                for item in response.get('files', []):
                    if len(rows) >= max_entries:
                        partial = True
                        break
                    item['relative_path'] = prefix + item['name']
                    rows.append(item)
                    if item['mimeType'] == FOLDER_MIME:
                        queue.append((item['id'], item['relative_path'] + '/'))
                token = response.get('nextPageToken')
                if not token or partial:
                    break
        except Exception as exc:
            errors.append({'folder_id': folder, 'error': str(exc)})
    return {'items': rows, 'partial': partial or bool(errors), 'errors': errors,
            'folders_scanned': len(visited)}

raw_inventory = inventory(RAW_FOLDER)
print('Raw entries:', len(raw_inventory['items']), 'partial:', raw_inventory['partial'])
print('Inventory errors:', raw_inventory['errors'])
print('Small PCAP candidates:', sum(
    x['name'].lower().endswith(('.pcap', '.pcapng')) and int(x.get('size', 0)) > 0
    and int(x['size']) <= 8 * 1024**2 for x in raw_inventory['items']))


In [ ]:
# Download the six exact baseline NPZs from their verified folder.
EXPECTED = {'USTC_1c_train.npz', 'USTC_1c_test.npz', 'mal_32_1c_train.npz',
            'mal_32_1c_test.npz', 'combined_train_data.npz', 'combined_test_data.npz'}
data_inventory = inventory(DATA_FOLDER, max_entries=100, max_folders=1)
items = [x for x in data_inventory['items'] if x['name'] in EXPECTED]
assert len(items) == 6 and {x['name'] for x in items} == EXPECTED, 'Missing/duplicate NPZ: inspect folder, do not substitute silently.'
assert all(x.get('size') for x in items), 'Unknown file sizes'
assert sum(int(x['size']) for x in items) <= 400 * 1024**2, 'NPZ download budget exceeded'

def download(item, directory, limit):
    size = int(item.get('size', 0))
    if not 0 < size <= limit:
        raise ValueError('Missing size or download limit exceeded: ' + item['name'])
    # Drive ID as prefix prevents filename collisions and path traversal.
    path = directory / (item['id'] + '_' + Path(item['name']).name)
    path.parent.mkdir(parents=True, exist_ok=True)
    request = drive.files().get_media(fileId=item['id'], supportsAllDrives=True)
    digest = hashlib.md5()
    with path.open('wb') as handle:
        loader = MediaIoBaseDownload(handle, request, chunksize=1024**2)
        done = False
        while not done:
            _, done = loader.next_chunk(num_retries=3)
            if handle.tell() > size or handle.tell() > limit:
                raise ValueError('File grew during transfer; stop and rerun audit')
    assert path.stat().st_size == size, 'Incomplete download'
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024**2), b''):
            digest.update(block)
    if item.get('md5Checksum'):
        assert digest.hexdigest() == item['md5Checksum'], 'Drive checksum mismatch'
    return path

npz_paths, download_errors = [], []
for item in sorted(items, key=lambda x: x['name']):
    try:
        npz_paths.append(download(item, WORK / 'npz', 400 * 1024**2))
        print('NPZ OK:', item['name'])
    except Exception as exc:
        download_errors.append({'file': item['name'], 'error': str(exc)})


In [ ]:
# Round-robin by source dataset; not a representative/random statistical sample.
groups = {}
for item in raw_inventory['items']:
    if item['name'].lower().endswith(('.pcap', '.pcapng')) and 0 < int(item.get('size', 0)) <= 8 * 1024**2:
        source = item['relative_path'].split('/')[0]
        groups.setdefault(source, []).append(item)
queues = [deque(sorted(items, key=lambda x: (int(x['size']), x['relative_path'])))
          for _, items in sorted(groups.items())]
selected = []
while queues and len(selected) < 6:
    for queue in queues:
        if queue and len(selected) < 6:
            selected.append(queue.popleft())
    queues = [queue for queue in queues if queue]
pcap_paths = []
for item in selected:
    try:
        pcap_paths.append(download(item, WORK / 'pcap', 8 * 1024**2))
    except Exception as exc:
        download_errors.append({'file': item['name'], 'error': str(exc)})
report = audit_paths(npz_paths, pcap_paths, packet_limit=10000)
report.update(run_id=RUN_ID, source_sha256=SOURCE_SHA256,
    source_folder_ids={'dataset': DATA_FOLDER, 'raw': RAW_FOLDER},
    raw_inventory=raw_inventory, dataset_inventory=data_inventory,
    selected_pcaps=selected, download_errors=download_errors,
    missing_dataset_count=6-len(npz_paths))
REPORT_PATH = WORK / ('pilot_audit_' + RUN_ID + '.json')
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
for item in report['datasets']:
    print('NPZ:', item['file'], '| rows:', item['samples'], '| keys:', item['keys'])
for item in report['pcaps']:
    print('PCAP:', item['file'], '|', item['status'])
print('Errors:', report['errors'] + download_errors)
print('Missing datasets:', report['missing_dataset_count'])
print('Full pairing verified:', report['training_ready'])
if not selected:
    print('No small PCAP selected in bounded scan. Check inventory/archives/permissions; this does NOT prove raw data absent.')
print('Local report:', REPORT_PATH)
print('NEXT: verify complete pairing / sessionization; no GPU training yet.')


## Simpan laporan ke folder pilot Drive A
Cell berikut hanya mengunggah laporan JSON baru, bukan checkpoint/PCAP/dataset.
Run ulang cell ini menggunakan file laporan yang sama tidak membuat duplikat.
Jika izin menulis ditolak, laporan lokal masih tersedia dan bisa diunduh manual.


In [ ]:
name = REPORT_PATH.name
existing = drive.files().list(q=f"'{PILOT_FOLDER}' in parents and trashed=false and name='{name}'",
    fields='files(id,name,webViewLink)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])
if existing:
    print('Laporan run ini sudah ada; tidak ditimpa:', existing)
else:
    uploaded = drive.files().create(body={'name': name, 'parents': [PILOT_FOLDER]},
        media_body=MediaFileUpload(str(REPORT_PATH), mimetype='application/json', resumable=True),
        fields='id,name,webViewLink', supportsAllDrives=True).execute()
    print('Tersimpan di folder PILOT_TEMPORAL_FUSION:', uploaded)
